In [0]:
from pyspark.sql import functions as F

country_currency_map = spark.createDataFrame([
    ("United States", "USD"), ("Canada", "CAD"), ("United Kingdom", "GBP"),
    ("France", "EUR"), ("Germany", "EUR"), ("Japan", "JPY"), ("Argentina", "BRL"),
    ("Brazil", "BRL"), ("Australia", "AUD"), ("China", "CNY"), ("India", "INR"),
    ("Mexico", "MXN"), ("South Korea", "KRW"), ("South Africa", "ZAR"),
    ("Switzerland", "CHF"), ("Sweden", "SEK"), ("Norway", "NOK"), ("Denmark", "DKK"),
    ("Poland", "PLN"), ("Turkey", "TRY"), ("Indonesia", "IDR"), ("Malaysia", "MYR"),
    ("Philippines", "PHP"), ("Thailand", "THB"), ("Israel", "ILS"), ("Hungary", "HUF"),
    ("Czechia", "CZK"), ("Romania", "RON"), ("Iceland", "ISK"), ("Singapore", "SGD"),
    ("Hong Kong", "HKD"), ("New Zealand", "NZD"),
], ["country_name", "transaction_currency"])

display(country_currency_map)

In [0]:
geo_df = spark.table("fraud_detection.silver.transactions_geo_enriched")

unmapped = (
    geo_df.select("country_name").distinct()
    .join(country_currency_map, on="country_name", how="left_anti")
)
display(unmapped)
print(f"Distinct countries in data: {geo_df.select('country_name').distinct().count()}")
print(f"Unmapped countries: {unmapped.count()}")

In [0]:
import requests, re

url = "https://raw.githubusercontent.com/thiagodp/country-to-currency/master/index.ts"
resp = requests.get(url)
resp.raise_for_status()

# parse lines like:  US: 'USD',
pattern = re.compile(r"^\s*([A-Z]{2}):\s*'([A-Z]{3})',?\s*$", re.MULTILINE)
matches = pattern.findall(resp.text)

print(f"Parsed {len(matches)} country-currency pairs")
print(matches[:5])

In [0]:
from pyspark.sql import functions as F

country_currency_df = spark.createDataFrame(matches, ["country_iso_code", "transaction_currency"])

country_currency_df.write.mode("overwrite").saveAsTable("fraud_detection.bronze.country_currency_map_raw")

display(country_currency_df.limit(10))
print(f"Row count: {country_currency_df.count()}")

In [0]:
geo_df = spark.table("fraud_detection.silver.transactions_geo_enriched")

geo_with_currency = geo_df.join(
    country_currency_df,
    on="country_iso_code",
    how="left"
)

unmapped_count = geo_with_currency.filter(F.col("transaction_currency").isNull()).count()
print(f"Transactions with no currency mapped: {unmapped_count}")

In [0]:
unmapped_df = geo_with_currency.filter(F.col("transaction_currency").isNull())

display(
    unmapped_df
    .groupBy("country_iso_code", "country_name")
    .count()
    .orderBy(F.desc("count"))
)

In [0]:
geo_with_currency_final = geo_with_currency.withColumn(
    "currency_is_defaulted",
    F.when(F.col("transaction_currency").isNull(), True).otherwise(False)
).withColumn(
    "transaction_currency",
    F.when(F.col("transaction_currency").isNull(), F.lit("USD")).otherwise(F.col("transaction_currency"))
)

display(geo_with_currency_final.groupBy("currency_is_defaulted").count())

In [0]:
from pyspark.sql import Window

fx_rates_df = spark.table("fraud_detection.bronze.fx_rates_raw")

# Get distinct transaction dates (just the date part, not the timestamp) and currencies needed
tx_dates_currencies = (
    geo_with_currency_final
    .withColumn("tx_date", F.to_date("transaction_date"))
    .select("tx_date", "transaction_currency")
    .distinct()
)

print(f"Distinct (date, currency) pairs needed: {tx_dates_currencies.count()}")

In [0]:
# Cross join each currency's available dates against tx_dates, keep only rate_date <= tx_date, then take the max
joined_candidates = tx_dates_currencies.join(
    fx_rates_df,
    (tx_dates_currencies.transaction_currency == fx_rates_df.currency) &
    (fx_rates_df.rate_date <= tx_dates_currencies.tx_date),
    "left"
)

window_spec = Window.partitionBy("tx_date", "transaction_currency").orderBy(F.desc("rate_date"))

fx_asof = (
    joined_candidates
    .withColumn("rn", F.row_number().over(window_spec))
    .filter(F.col("rn") == 1)
    .select(
        F.col("tx_date"),
        F.col("transaction_currency"),
        F.col("rate_date").alias("fx_rate_date_used"),
        F.col("rate_to_usd")
    )
)

display(fx_asof.limit(10))
print(f"Row count: {fx_asof.count()}")

In [0]:
total_pairs = tx_dates_currencies.count()
unresolved_pairs = fx_asof.filter(F.col("rate_to_usd").isNull()).count()
unresolved_currencies = (
    fx_asof.filter(F.col("rate_to_usd").isNull())
    .select("transaction_currency").distinct()
)

print(f"Total (date, currency) pairs: {total_pairs}")
print(f"Unresolved pairs: {unresolved_pairs} ({unresolved_pairs / total_pairs:.1%})")
print(f"Distinct currencies with no FX coverage: {unresolved_currencies.count()}")
display(unresolved_currencies)

In [0]:
usd_impact = (
    geo_with_currency_final
    .filter(F.col("transaction_currency") == "USD")
    .count()
)
print(f"Transactions in USD: {usd_impact} ({usd_impact / 1472952:.1%})")

In [0]:
unresolved_currency_list = [row.transaction_currency for row in unresolved_currencies.collect()]

impacted_transactions = (
    geo_with_currency_final
    .filter(F.col("transaction_currency").isin(unresolved_currency_list))
    .count()
)
print(f"Actual transactions affected by unresolved FX currencies: {impacted_transactions} ({impacted_transactions / 1472952:.1%})")

# breakdown by currency, ordered by how much it actually matters
display(
    geo_with_currency_final
    .filter(F.col("transaction_currency").isin(unresolved_currency_list))
    .groupBy("transaction_currency")
    .count()
    .orderBy(F.desc("count"))
)

In [0]:
impacted_v2 = (
    geo_with_currency_final
    .filter(F.col("transaction_currency").isin(["XCG", "ZWG"]))
    .count()
)
print(f"Transactions affected by XCG/ZWG: {impacted_v2} ({impacted_v2 / 1472952:.2%})")

In [0]:
fx_asof_final = fx_asof.withColumn(
    "fx_rate_is_defaulted",
    F.when(F.col("rate_to_usd").isNull(), True).otherwise(False)
).withColumn(
    "rate_to_usd",
    F.when(F.col("rate_to_usd").isNull(), F.lit(1.0)).otherwise(F.col("rate_to_usd"))
)

display(fx_asof_final.groupBy("fx_rate_is_defaulted").count())

In [0]:
transactions_with_date = geo_with_currency_final.withColumn("tx_date", F.to_date("transaction_date"))

silver_fx_enriched = transactions_with_date.join(
    fx_asof_final,
    (transactions_with_date.tx_date == fx_asof_final.tx_date) &
    (transactions_with_date.transaction_currency == fx_asof_final.transaction_currency),
    "left"
).drop(fx_asof_final.tx_date).drop(fx_asof_final.transaction_currency).withColumn(
    "transaction_amount_usd", F.col("transaction_amount") * F.col("rate_to_usd")
)

print(f"Row count: {silver_fx_enriched.count()}")
display(silver_fx_enriched.select("transaction_id", "transaction_amount", "transaction_currency", "rate_to_usd", "transaction_amount_usd").limit(10))

In [0]:
silver_fx_enriched = silver_fx_enriched.withColumn(
    "transaction_amount_usd", F.col("transaction_amount") / F.col("rate_to_usd")
)

display(silver_fx_enriched.select("transaction_id", "transaction_amount", "transaction_currency", "rate_to_usd", "transaction_amount_usd").limit(10))

In [0]:
final_fx_columns = [
    "transaction_id", "customer_id", "transaction_amount", "transaction_currency",
    "transaction_amount_usd", "rate_to_usd", "fx_rate_is_defaulted", "transaction_date",
    "payment_method", "product_category", "quantity", "customer_age", "customer_location",
    "device_used", "ip_address", "shipping_address", "billing_address", "is_fraudulent",
    "account_age_days", "transaction_hour", "country_name", "country_iso_code",
    "subdivision_1_name", "city_name", "time_zone", "currency_is_defaulted"
]

silver_fx_final = silver_fx_enriched.select(*final_fx_columns)

silver_fx_final.write.mode("overwrite").saveAsTable("fraud_detection.silver.transactions_fx_enriched")

display(spark.sql("SELECT COUNT(*) FROM fraud_detection.silver.transactions_fx_enriched"))